# YouTube Auto-Dub — dub + lip-sync on a free Colab GPU

Runtime → **Change runtime type → GPU (T4)**. This dubs a YouTube video into
another language *in the original voice* and (optionally) re-renders the mouth to
match, using Wav2Lip. Everything is open-source and free.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Install ffmpeg + ytdub
The `[xtts,nllb]` extras pull the cloning TTS and the neural translator. On the
Colab GPU runtime torch already has CUDA, so torchcodec loads fine here.

In [ ]:
!apt-get -qq install -y ffmpeg
!pip -q install "ytdub[chatterbox,nllb] @ git+https://github.com/mazzasaverio/youtube-auto-dub.git@master"

## 3. Pick a video and dub it (no lip-sync yet)
First run downloads the models (~4 GB). Output lands in `data/output/`.

In [ ]:
import os
os.environ['COQUI_TOS_AGREED'] = '1'      # accept the XTTS license
URL = 'https://www.youtube.com/shorts/fc4BozJTvc4'  # <-- change me
!ytdub dub "$URL" --target en --asr-model medium --translator nllb

## 4. Set up Wav2Lip (its own venv — its deps conflict with coqui-tts)
We create a separate virtualenv for Wav2Lip and download its checkpoint. If the
checkpoint URL 404s, grab `wav2lip_gan.pth` from the Wav2Lip README and drop it in
`Wav2Lip/checkpoints/`.

In [ ]:
%%bash
set -e
cd /content
[ -d Wav2Lip ] || git clone -q https://github.com/Rudrabha/Wav2Lip
cd Wav2Lip
python3 -m venv .venv
./.venv/bin/pip -q install --upgrade pip
# Wav2Lip pins old libs; these versions are known to work on Colab:
./.venv/bin/pip -q install numpy==1.23.5 opencv-python librosa==0.9.2 numba==0.58.1 \
    torch torchvision tqdm 'scipy<1.13'
mkdir -p checkpoints face_detection/detection/sfd
# face detector + generator checkpoints (swap the URLs if they move):
wget -q -O face_detection/detection/sfd/s3fd.pth \
  https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth || true
wget -q -O checkpoints/wav2lip_gan.pth \
  https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth || true
ls -lh checkpoints/

## 5. Dub **with lip-sync**
Point ytdub at the Wav2Lip venv and re-run with `--lipsync`. Wav2Lip re-renders the
mouth to match the dubbed audio; the result is remuxed as a share-ready MP4.

In [ ]:
import os
os.environ['YTDUB_WAV2LIP_DIR'] = '/content/Wav2Lip'
os.environ['YTDUB_WAV2LIP_CKPT'] = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
os.environ['YTDUB_WAV2LIP_PYTHON'] = '/content/Wav2Lip/.venv/bin/python'
!ytdub dub "$URL" --target en --asr-model medium --translator nllb --lipsync --subtitles

## 6. Preview / download the result

In [ ]:
import glob
from IPython.display import Video, display
out = sorted(glob.glob('data/output/*.mp4'))[-1]
print(out)
display(Video(out, embed=True, width=360))
# from google.colab import files; files.download(out)